In [0]:
orders_schema = "order_id long, order_date string, cust_id long, order_status string"

In [0]:
orders_df = spark.read.format("csv").schema(orders_schema).load('/Volumes/azure_data_engineering_learn_ws/default/datasets')

In [0]:
orders_df.createOrReplaceTempView("orders")

In [0]:
spark.sql("select count(*) from orders where order_status = 'CLOSED'").show()

In [0]:
part_df = orders_df.write.mode("overwrite").format("delta").partitionBy("order_status").saveAsTable("azure_data_engineering_learn_ws.default.orders_partition_output")

In [0]:
for f in spark.table("azure_data_engineering_learn_ws.default.orders_partition_output").inputFiles():
  print(f)

In [0]:
%sql
DESCRIBE EXTENDED azure_data_engineering_learn_ws.default.orders_partition_output

In [0]:
# Read entire partitioned table
partitioned_df = spark.read.table("azure_data_engineering_learn_ws.default.orders_partition_output")
print("Total rows:", partitioned_df.count())
partitioned_df.printSchema()

# Read only a specific partition (partition pruning — reads only that folder)
closed_df = spark.read.table("azure_data_engineering_learn_ws.default.orders_partition_output") \
    .filter("order_status = 'CLOSED'")
print("CLOSED rows:", closed_df.count())
display(closed_df)

In [0]:
new_df = spark.read.table("azure_data_engineering_learn_ws.default.orders_partition_output")

In [0]:
new_df.select("order_status").distinct().show()

In [0]:
new_df.createOrReplaceTempView("orders_partition")

In [0]:
spark.sql("select count(*) from orders_partition where order_status = 'COMPLETE'").show()

In [0]:
spark.sql("select count(*) from orders_partition where order_status = 'PENDING_PAYMENT'").show()